#***Pipeline* de PLN para el Corpus Oral de Referencia del Español en Contacto (COREC): etiquetado automático POS/NER, alineamiento determinista O(1) mediante clave triple estable y generación de CoNLL-U Plus**

Este cuaderno implementa un *pipeline* de procesamiento del lenguaje natural (PLN) desarrollado en el marco del Corpus Oral de Referencia del Español en Contacto (COREC), Fase II: lenguas minoritarias.

Su objetivo principal es integrar el etiquetado lingüístico automático con un mecanismo de alineamiento determinista que mantenga la correspondencia entre las formas originales presentes en las transcripciones ortográficas normalizadas del corpus y los tokens generados por la librería de anotación automática Stanza.

El proceso comienza con el etiquetado automático (POS *tagging*) y de reconocimiento de entidades nombradas (NER), según el estándar *Universal Dependencies* (UD) con salida en formato CoNLL-U. Sobre el *output* se aplica un algoritmo de alineamiento determinista que emplea una clave triple estable que identifica de manera unívoca cada token y preserva la trazabilidad entre las representaciones originales y normalizadas del corpus. A partir de esta clave se construyen índices que permiten consultas en tiempo constante (O(1)).

Finalmente, el *pipeline* genera archivos en formato CoNLL-U Plus, una extensión del formato CoNLL-U que posibilita la incorporación de columnas adicionales de anotación. En concreto, se han añadido las columnas COREC:NER, COREC:DISFL y COREC:FORM_ORIG para posteriores análisis y consultas lingüísticas.

#**PARTE I**

##**Preprocesamiento y anotación automática POS y NER con Stanza**

En esta primera parte se ha aplicado un *pipeline* de Procesamiento del Lenguaje Natural (PLN) mediante Stanza (Stanford NLP) que ejecuta tokenización, POS *tagging*, lematización y *Named Entity Recognitio*n (NER), con salidas en formato CoNLL-U.

###**I. a) Instalación y dependencias**

Instalación de la librería de PLN Stanza y descarga del modelo en español.

In [ ]:
# ============================================================
# INSTALACION__STANZA
# ============================================================

!pip install stanza -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 14.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# MODELO__STANZA__DESCARGA_ES
# ============================================================

import stanza
stanza.download("es")

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: es (Spanish) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/es/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

###**I. b) Carga de archivos e importaciones**

In [ ]:
# ============================================================
# SUBIR_ARCHIVOS_TXT__COLAB
# ============================================================

from google.colab import files
uploaded = files.upload()

Saving 001_003_001_002_002_012_002_seg_normas_1_normas_2 (15).txt to 001_003_001_002_002_012_002_seg_normas_1_normas_2 (15).txt
Saving 002_007_001_002_002_002_006_seg_normas_1_normas_2 (6).txt to 002_007_001_002_002_002_006_seg_normas_1_normas_2 (6).txt
Saving 003_006_004_001_001_029_006_seg_normas_1_normas_2 (3).txt to 003_006_004_001_001_029_006_seg_normas_1_normas_2 (3).txt
Saving 004_006_001_001_002_032_001_seg_normas_1_normas_2 (15).txt to 004_006_001_001_002_032_001_seg_normas_1_normas_2 (15).txt
Saving 005_004_002_003_002_001_001_seg_normas_1_normas_2 (3).txt to 005_004_002_003_002_001_001_seg_normas_1_normas_2 (3).txt
Saving 008_001_001_001_001_033_001_seg_normas_1_normas_2 (9).txt to 008_001_001_001_001_033_001_seg_normas_1_normas_2 (9).txt
Saving 010_006_002_001_002_014_008_seg_normas_1_normas_2 (5).txt to 010_006_002_001_002_014_008_seg_normas_1_normas_2 (5).txt
Saving 011_005_002_001_001_002_012_seg_normas_1_normas_2 (1).txt to 011_005_002_001_001_002_012_seg_normas_1_norma

In [ ]:
# ============================================================
# IMPORTACIONES_Y_LECTURA_INICIAL
# ============================================================
import re
import glob
from pathlib import Path
archivos = glob.glob("/content/*.txt")
content = Path(archivos[0]).read_text(encoding="utf-8")
from stanza.utils.conll import CoNLL

###**I. c) Preparación de textos**

Luego, se han preparado los textos para su procesamiento con Stanza.
Se han eliminado líneas vacías y se han normalizado espacios en blanco. Cada línea del archivo se ha considerado una oración independiente.

In [ ]:
# ============================================================
# ENTRADA__1_LINEA_1_ORACION
# ============================================================

def build_text_from_lines(content: str) -> tuple[list[str], str]:
    """Prepara texto 1 línea=1 oración para Stanza: devuelve (sents, text con \n\n)."""

    # 1) Filtra líneas vacías y elimina espacios en blanco en los extremos
    sents = [line.strip() for line in content.splitlines() if line.strip()]

    # 2) Une con doble salto para preservar fronteras de oración
    text = "\n\n".join(sents)

    return sents, text


###**I. d) Construcción del *pipeline* de Stanza**

Se ha definido el *pipeline* de Stanza para español con la activación de
los módulos de tokenización, *multi-word tokens* (MWT), etiquetado morfosintáctico (POS), lematización y reconocimiento de entidades (NER).
La segmentación discursiva no se ha realizado automáticamente, sino que se preserva la segmentación establecida en la entrada,

In [ ]:
# ============================================================
# PIPELINE__STANZA__DEFAULT_POS_LEMMA__NER
# ============================================================

def build_stanza_pipeline() -> stanza.Pipeline:
    return stanza.Pipeline(
        "es",
        processors="tokenize,mwt,pos,lemma,ner",
        tokenize_no_ssplit=True,
        use_gpu=True,
        verbose=False,
    )

###**I. e) Configuración de la salida en formato CoNLL-U**

Después, se ha configurado el directorio donde se guardarán los resultados y se ha establecido la ruta de salida para cada archivo procesado.
Las salidas se generarán en formato CoNLL-U, estándar de *Universal Dependencies*.

In [ ]:
# ============================================================
# SALIDA__CONLL_U__1TXT_1CONLLU
# ============================================================
OUT_DIR = Path("/content/conllu_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def out_conllu_path(input_path: Path):
    p = Path(input_path)
    return OUT_DIR / f"{p.stem}.conllu"

### **I. f) Ejecución por lotes y exportación**

Se ha aplicado el *pipeline* de Stanza a cada archivo .txt, se ha comprobado la preservación de la segmentación de la entrada y se ha guardado la salida en un archivo .conllu independiente por entrada.

In [ ]:
# ============================================================
# EJECUCION__STANZA__TXTs__EXPORTACIÓN_CADA_UNO_CONLLU
# ============================================================

# Define 'archivos' using the keys from the 'uploaded' dictionary
archivos = [f"/content/{filename}" for filename in uploaded.keys()]

nlp = build_stanza_pipeline()

for i, path in enumerate(archivos, start=1):
    content = Path(path).read_text(encoding="utf-8")
    sents, text = build_text_from_lines(content)

    doc = nlp(text)
    assert len(doc.sentences) == len(sents)

    CoNLL.write_doc2conll(doc, str(out_conllu_path(path)))

    print("Procesados", i, "archivos")

Procesados 1 archivos
Procesados 2 archivos
Procesados 3 archivos
Procesados 4 archivos
Procesados 5 archivos
Procesados 6 archivos
Procesados 7 archivos
Procesados 8 archivos
Procesados 9 archivos
Procesados 10 archivos
Procesados 11 archivos


#**PARTE II**

##**Algoritmo de alineamiento basado en triple clave jerárquica e índice O(1)**

El alineamiento entre las formas resultantes del proceso de normalización y los tokens generados por Stanza se ha realizado mediante un algoritmo determinista basado en una clave triple estable: (`id_archivo, id_ud, id_conllu`) en la que el primer elemento de la tupla es el identificador de archivo, el segundo el de la unidad discursiva y  el tercero al identificador del token en la salida CoNLL-U.

El procedimiento se ha basado en:

1. Construcción de índices de tokens por oración.
2. Identificación de tokens individuales mediante consultas O(1).
3. Resolución de secuencias de tokens (n-gramas).
4. Tratamiento específica de clíticos.

Este enfoque preserva la trazabilidad entre las distintas representaciones del texto y resuelve la correspondencia entre las formas normalizadas, las originales y las generadas por Stanza en la salida CoNLL-U.

###**II.a) Configuración y extracción del identificador de archivo**

Tras haberse importado las librerías necesarias y  haberse definido el directorio donde están los archivos CoNLL-U que servirán como entrada del algoritmo, se ha creado la función que extrae el identificador base del archivo mediante una expresión regular. Este identificador formará parte de la clave estable del algoritmo de alineamiento.

In [ ]:
# ============================================================
# CONFIG_Y_REGEX
# ============================================================
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd

CONLLU_DIR = Path("/content/conllu_out")

_ID_BASE_RE = re.compile(r"^(?:\d+_){6}\d+")

In [ ]:
def id_archivo_base_from_conllu_path(p: Path) -> str:
    """
    Extrae el `id_archivo_base` desde el nombre del archivo (Path.stem) usando
    el patrón `_ID_BASE_RE`.

    Parameters
    ----------
    p : Path
        Ruta al archivo (normalmente un .conllu / .conllup). Se usa `p.stem`
        (nombre sin extensión) como cadena de entrada para la regex.

    Returns
    -------
    str
        Identificador base del archivo (match completo: `m.group(0)`).

    Raises
    ------
    ValueError
        Si `p.stem` no cumple el patrón `_ID_BASE_RE` y no se puede extraer
        el identificador.
    """
    m = _ID_BASE_RE.match(p.stem)
    if not m:
        raise ValueError(f"No puedo extraer id_archivo base de: {p.name}")
    return m.group(0)

###**II.b) Parser CoNLL-U y extracción de tokens**

Posteriormente, se ha parseado el archivo CoNLL-U para reconstruir las unidades discursivas y extraer los tokens. El resultado se ha organizado en una estructura de datos en Python que representa cada oración mediante un identificador de unidad discursiva (`id_ud`) y la lista de tokens que la componen. El identificador `id_ud` se ha reconstruido a partir del campo `# sent_id` presente en el archivo CoNLL-U.

Posteriormente, cada token se ha almacenado como un par formado por (`id_conllu, FORM`), en el que `id_conllu` se corresponde con el identificador numérico del token dentro de la oración y FORM con la palabra tal como aparece en la segunda columna de la salida CoNLL-U.
Durante el proceso se  han ignorado los comentarios, los Multi-Word Tokens y los *empty nodes*.

In [ ]:
# ============================================================
# PARSER_CONLLU_A_ORACIONES
# Devuelve: [(id_ud, [(id_conllu, FORM), ...]), ...]
# ============================================================
Token = Tuple[str, str]            # (id_conllu, FORM)
SentenceTokens = Tuple[str, List[Token]]  # (id_ud, tokens)

def conllu_to_sentence_tokens(conllu_text: str) -> List[SentenceTokens]:
    """
    Parsea CoNLL-U y devuelve tokens reales por oración: (id_ud, [(ID, FORM), ...]).
    Segmenta por líneas en blanco y crea id_ud desde '# sent_id' (0->UD00001).
    Ignora comentarios, filas no estándar, MWT (3-4) y empty nodes (2.1).
    """
    ...
    out: List[SentenceTokens] = []
    current_ud: Optional[str] = None
    tokens: List[Token] = []

    for line in conllu_text.splitlines():
        if not line.strip():
            if current_ud is not None:
                out.append((current_ud, tokens))
            current_ud, tokens = None, []
            continue

        if line.startswith("# sent_id"):
            sent = line.split("=", 1)[1].strip()
            current_ud = f"UD{(int(sent) + 1):05d}" if sent.isdigit() else None
            continue

        if current_ud is None or line.startswith("#"):
            continue

        cols = line.split("\t")
        if len(cols) != 10:
            continue

        ID, FORM = cols[0], cols[1]

        # Ignorar MWT (3-4) y empty nodes (2.1) y cualquier ID no entero
        if "-" in ID or "." in ID or (not ID.isdigit()):
            continue

        tokens.append((ID, FORM))

    if current_ud is not None:
        out.append((current_ud, tokens))

    return out

###**II.c) Inventario de tokens por unidad discursiva**
Luego, se ha creado un índice global en memoria que organiza los tokens extraídos de los archivos CoNLL-U por unidad discursiva. Cada entrada del diccionario utiliza como clave el par (`id_archivo`, `id_ud`), que identifica una unidad discursiva dentro de un archivo. El valor asociado a cada clave es la lista de tokens de ese enunciado, representados como pares (`id_conllu`, `FORM`). De este modo, se pueden recuperar los tokens de cualquier unidad discursiva mediante su clave estable en tiempo constante (O(1)).

In [ ]:
# ============================================================
# SENT_MAP
# sent_map[(id_archivo, id_ud)] = [(id_conllu, FORM), ...]
# ============================================================
SentKey = Tuple[str, str]  # (id_archivo, id_ud)
SentMap = Dict[SentKey, List[Token]]

def build_sent_map(conllu_dir: Path) -> SentMap:
    """
    Construye un diccionario global en memoria que indexa los tokens reales
    por oración a partir de archivos `.conllu`.

    Clave: (id_archivo, id_ud)
    Valor: [(id_conllu, FORM), ...]

    Permite acceso O(1) a los tokens de cualquier oración mediante la clave.
    """
    sent_map: SentMap = {}

    for conllu_path in sorted(conllu_dir.glob("*.conllu")):
        id_archivo = id_archivo_base_from_conllu_path(conllu_path)
        text = conllu_path.read_text(encoding="utf-8")

        for id_ud, toks in conllu_to_sentence_tokens(text):
            sent_map[(id_archivo, id_ud)] = toks

    return sent_map

sent_map = build_sent_map(CONLLU_DIR)
print("OK sent_map oraciones:", len(sent_map))

# ejemplo
if sent_map:
    k = next(iter(sent_map))
    print("EJ clave:", k)
    print("EJ primeros tokens:", sent_map[k][:10])

OK sent_map oraciones: 6167
EJ clave: ('001_003_001_002_002_012_002', 'UD00001')
EJ primeros tokens: [('1', 'bueno'), ('2', 'para'), ('3', 'fuera'), ('4', 'de'), ('5', 'el'), ('6', 'país'), ('7', 'no'), ('8', 'pero'), ('9', 'dentro'), ('10', 'de')]


###**II. d) Configuración de entradas y salidas y carga del *merged***
A partir de `sent_map` (inventario de tokens CoNLL-U por `(id_archivo, id_ud)`), se ha preparado el proceso de alineamiento con el archivo `Log_4_9_11_merged.csv`, que contiene las columnas `forma_original` y `forma_resultante` (correspondientes, respectivamente, a la variante léxica original y a la generada durante el proceso de normalización de las reglas 4, 9 y 11), con el objetivo de añadir la columna `id_conllu`.

Se han definido después `MERGED_PATH`, que apunta al log fusionado de la normalización filtrada (normas 4, 9 y 11), y `OUT_PATH`, que define el archivo de salida donde se guardará ese mismo log enriquecido con la columna `id_conllu` tras el alineamiento.

A continuación, el archivo `Log_4_9_11_merged.csv` se ha cargado en memoria en un DataFrame de pandas, `merged`, que será recorrido posteriormente para aplicar el proceso de alineamiento.

In [ ]:
# ============================================================
# CONFIG
# ============================================================

MERGED_PATH = Path("/content/Log_4_9_11_merged.csv")
OUT_PATH    = Path("/content/Log_4_9_11_merged_con_id_conllu.csv")

In [ ]:
# ============================================================
# LOAD_MERGED
# ============================================================
merged = pd.read_csv(MERGED_PATH, sep=";", dtype=str).fillna("")
print("OK merged filas:", len(merged), "cols:", list(merged.columns))

OK merged filas: 2260 cols: ['id_archivo', 'id_ud', 'linea_n', 'hablante', 'rol', 'norma_id', 'fenomeno', 'forma_original', 'forma_resultante', 'accion', 'contexto']


###**II.e) Índice token a token por ocurrencia O(1)**

A partir de `sent_map`, que contiene el inventario de tokens CoNLL-U por unidad discursiva, se ha construido un índice para localizar de forma unívoca cada token dentro de su enunciado a partir de la columna FORM del archivo CoNLL-U y de su número de aparición.

Para cada unidad discursiva identificada por (`id_archivo, id_ud`), se recorre la lista de tokens generados en el archivo CoNLL-U y se cuenta cuántas veces aparece cada FORM. Con esta información se crea una clave compuesta (`id_archivo, id_ud, FORM, ocurrencia_en_oracion`), que devuelve el `id_conllu` correspondiente.

Este índice resuelve la correspondencia entre las formas presentes en el archivo `Log_4_9_11_merged.csv` y los tokens del análisis automático mediante consultas en tiempo constante promedio O(1).

In [ ]:
# ============================================================
# IDX_FORM_OCC__O1
# Construye un índice O(1) por unidad discursiva:
# (id_archivo, id_ud, FORM, ocurrencia_en_ud) -> id_conllu
# donde counts[form] es la 1ª/2ª/3ª vez que aparece FORM en esa unidad discursiva.
# ============================================================

IdxKey = Tuple[str, str, str, int]
IDX_FORM_OCC: Dict[IdxKey, str] = {}

for (id_archivo, id_ud), toks in sent_map.items():
    counts = defaultdict(int)
    for id_conllu, form in toks:
        counts[form] += 1
        IDX_FORM_OCC[(id_archivo, id_ud, form, counts[form])] = id_conllu
s
print("OK IDX_FORM_OCC keys:", len(IDX_FORM_OCC))

OK IDX_FORM_OCC keys: 56080


###**II.f) Alineamiento O(1) para filas de 1 token**

Tras la construcción de `IDX_FORM_OCC`, se han recorrido las filas del DataFrame `merged` para intentar asignar el identificador `id_conllu` a cada fila cuando `forma_resultante` se corresponde con un único token.

Para resolver posibles ambigüedades en una unidad discursiva, se ha calculado la ocurrencia de `(id_archivo, id_ud, forma_resultante)` y se ha consultado el índice mediante la clave `(id_archivo, id_ud, FORM, ocurrencia_en_oracion)`. De este modo, se ha obtenido directamente el `id_conllu` del token correspondiente.

Si `forma_resultante` contiene espacios, la fila se ha marcado como `MULTI_TOKEN` para resolverla posteriormente mediante el procedimiento de alineamiento para secuencias de varios tokens.

El proceso registra, además, un estado por fila (`OK`, `NO_MATCH`, `MULTI_TOKEN`, etc.) que mantiene la trazabilidad del alineamiento.

In [ ]:
# ============================================================
# APPLY_MATCH PARA_1TOKEN
#   Para cada fila del merged:
#     - si forma_resultante es 1 token -> O(1) con ocurrencia
#     - si tiene espacios -> MULTI_TOKEN
# ============================================================

def is_multi_token(s: str) -> bool:
    return len([t for t in s.strip().split() if t]) > 1

merged_occ = defaultdict(int)

id_conllu_out = []
status_out = []

for a, u, fr in zip(merged["id_archivo"], merged["id_ud"], merged["forma_resultante"]):
    fr = str(fr).strip()
    if not a or not u:
        id_conllu_out.append("")
        status_out.append("NO_KEY")
        continue

    if not fr:
        id_conllu_out.append("")
        status_out.append("NO_FORMA_RESULTANTE")
        continue

    if is_multi_token(fr):
        id_conllu_out.append("")
        status_out.append("MULTI_TOKEN")  # <- se resuelve en un paso
        continue

    merged_occ[(a, u, fr)] += 1
    occ = merged_occ[(a, u, fr)]

    k = (a, u, fr, occ)
    idc = IDX_FORM_OCC.get(k)

    if idc is None:
        id_conllu_out.append("")
        status_out.append("NO_MATCH")
    else:
        id_conllu_out.append(idc)
        status_out.append("OK")

merged["id_conllu"] = id_conllu_out
merged["id_conllu_status"] = status_out

print(merged["id_conllu_status"].value_counts(dropna=False))

id_conllu_status
OK             2009
MULTI_TOKEN     234
NO_MATCH         17
Name: count, dtype: int64


###**II.g) Exportación del *merged* enriquecido**

Finalmente, se ha añadido al `Log_4_9_11_merged.csv` la columna `id_conllu`  con el resultado del alineamiento y se ha guardado el archivo de salida, `Log_4_9_11_merged_con_id_conllu.csv` en OUT_PATH.

In [ ]:
# ============================================================
# SAVE
# ============================================================
merged.to_csv(OUT_PATH, sep=";", index=False, encoding="utf-8-sig")
print("OK ->", OUT_PATH)

OK -> /content/Log_4_9_11_merged_con_id_conllu.csv


###**II.h) Alineamiento de multitoken mediante n-gramas y ocurrencia (O(1) en consulta**

A continuación, se han resuelto las filas del DataFrame `merged` marcadas como MULTI_TOKEN. Para ello se ha construido un índice de n-gramas por unidad discursiva a partir de `sent_map`. La clave del índice es (`id_archivo, id_ud, ngram_tuple, ocurrencia`) y el valor devuelto es el span i-j (rango de `id_conllu`) que cubre ese n-grama en la oración.

Después, para cada fila MULTI_TOKEN, se ha tokenizado `forma_resultante`, se ha calculado su número de aparición en la unidad discursiva y se ha consultado el índice para asignar `id_conllu` como span (OK_2TOKEN, OK_3TOKEN).
Cuando no existe correspondencia, el estado correspondiente se ha registrado.

In [ ]:
# ============================================================
# IDX_NGRAM_OCC__O1
# IDX_NGRAM_OCC[(id_archivo, id_ud, ngram_tuple, occ)] = "i-j"
# (N se limita con NGRAM_MIN/MAX para no inflar memoria)
# ============================================================
NGRAM_MIN = 2
NGRAM_MAX = 3

Ngram = Tuple[str, ...]
NgramKey = Tuple[str, str, Ngram, int]
IDX_NGRAM_OCC: Dict[NgramKey, str] = {}

for (id_archivo, id_ud), toks in sent_map.items():
    forms = [w for _, w in toks]
    ids = [i for i, _ in toks]

    for n in range(NGRAM_MIN, NGRAM_MAX + 1):
        counts = defaultdict(int)
        for start in range(0, len(forms) - n + 1):
            ngram = tuple(forms[start:start + n])
            counts[ngram] += 1
            occ = counts[ngram]
            IDX_NGRAM_OCC[(id_archivo, id_ud, ngram, occ)] = f"{ids[start]}-{ids[start + n - 1]}"

print("OK IDX_NGRAM_OCC keys:", len(IDX_NGRAM_OCC))


# ============================================================
# APPLY_MATCH__O1_PARA_MULTI_TOKEN (MULTI_TOKEN*)
#   Resuelve 2..NGRAM_MAX tokens con ocurrencia
# ============================================================
multi_occ = defaultdict(int)

for i, row in merged.iterrows():
    st = str(row.get("id_conllu_status", "")).strip()
    if not st.startswith("MULTI_TOKEN"):
        continue

    a = str(row["id_archivo"]).strip()
    u = str(row["id_ud"]).strip()
    fr = str(row["forma_resultante"]).strip()

    parts = tuple(t for t in fr.split() if t)
    n = len(parts)

    if n < NGRAM_MIN or n > NGRAM_MAX:
        merged.at[i, "id_conllu_status"] = "MULTI_TOKEN_LEN_OUT_OF_RANGE"
        continue

    multi_occ[(a, u, parts)] += 1
    occ = multi_occ[(a, u, parts)]

    span = IDX_NGRAM_OCC.get((a, u, parts, occ))
    if span is None:
        merged.at[i, "id_conllu"] = ""
        merged.at[i, "id_conllu_status"] = f"NO_MATCH_{n}TOKEN"
    else:
        merged.at[i, "id_conllu"] = span
        merged.at[i, "id_conllu_status"] = f"OK_{n}TOKEN"

print(merged["id_conllu_status"].value_counts(dropna=False))

OK IDX_NGRAM_OCC keys: 94359
id_conllu_status
OK                 2009
OK_2TOKEN           215
NO_MATCH             17
NO_MATCH_2TOKEN      12
OK_3TOKEN             7
Name: count, dtype: int64


###**II.i) Resolución de clíticos**
Posteriormente, se ha aplicado un procedimiento de *fallback* sobre algunas filas marcadas previamente como `NO_MATCH`. Si `forma_resultante` presenta la estructura de una forma verbal con enclítico, la palabra se segmenta en dos partes (`stem`, `clitic`) y dicha secuencia se busca como bigrama en el índice `IDX_NGRAM_OCC`.

Cuando existe correspondencia, se asigna a `id_conllu` el *span* de los dos tokens y se actualiza el estado del alineamiento a `OK_CLITIC_2TOKEN`. Cuando no existe correspondencia, se registra el estado `NO_MATCH_CLITIC_2TOKEN`.

In [ ]:
# ============================================================
# FALLBACK_CLITICOS__DESDE_NO_MATCH__O1
#  Si forma_resultante es 1 token pero Stanza lo separó (VERBO + clítico),
#  intentamos resolver con IDX_NGRAM_OCC como bigrama (stem, clitic).
# ============================================================

CLITICOS = (
    "me","te","se","nos","os",
    "lo","la","los","las",
    "le","les"
)

def split_enclitic_2token(word: str) -> tuple[str, str] | None:
    w = word.strip()
    if not w:
        return None
    # preferir clíticos largos primero (les/los/las) para evitar cortar mal
    for cl in sorted(CLITICOS, key=len, reverse=True):
        if w.endswith(cl) and len(w) > len(cl):
            stem = w[:-len(cl)]
            return (stem, cl)
    return None

clitic_occ = defaultdict(int)

for i, row in merged.iterrows():
    if row.get("id_conllu_status") != "NO_MATCH":
        continue

    a = str(row["id_archivo"]).strip()
    u = str(row["id_ud"]).strip()
    fr = str(row["forma_resultante"]).strip()

    parts2 = split_enclitic_2token(fr)
    if parts2 is None:
        continue

    # Buscar como bigrama (stem, clitic) usando el mismo índice O(1)
    clitic_occ[(a, u, parts2)] += 1
    occ = clitic_occ[(a, u, parts2)]

    span = IDX_NGRAM_OCC.get((a, u, parts2, occ))
    if span is None:
        merged.at[i, "id_conllu"] = ""
        merged.at[i, "id_conllu_status"] = "NO_MATCH_CLITIC_2TOKEN"
    else:
        merged.at[i, "id_conllu"] = span
        merged.at[i, "id_conllu_status"] = "OK_CLITIC_2TOKEN"

print(merged["id_conllu_status"].value_counts(dropna=False))

id_conllu_status
OK                  2009
OK_2TOKEN            215
NO_MATCH_2TOKEN       12
OK_CLITIC_2TOKEN       9
NO_MATCH               8
OK_3TOKEN              7
Name: count, dtype: int64


###**II.j) Exportación del archivo final**

En este paso se ha guardado el DataFrame `merged` enriquecido con la columna `id_conllu` generada durante el proceso de alineamiento. El resultado se ha exportado como un nuevo archivo CSV (`Log_4_9_11_merged_con_id_conllu_final.csv`), que constituye la versión final del registro con la correspondencia entre las formas del log y los tokens —o secuencias de tokens— de la salida CoNLL-U.

In [ ]:
# ============================================================
# SAVE__MERGED_FINAL
# ============================================================

OUT_FINAL = Path("/content/Log_4_9_11_merged_con_id_conllu_final.csv")

merged.to_csv(OUT_FINAL, sep=";", index=False, encoding="utf-8-sig")

print("OK guardado en:", OUT_FINAL)

OK guardado en: /content/Log_4_9_11_merged_con_id_conllu_final.csv


###**II.k) Inspección de casos no resueltos**

En esta celda se han filtrado las filas del DataFrame `merged` cuyo estado es `NO_MATCH` y `NO_MATCH_2TOKEN`  y se ha generado una tabla para su revisión.
Asimismo, se han mostrado los diez primeros casos problemáticos junto con la secuencia de tokens de su unidad discursiva.

In [ ]:
# ============================================================
# 2__INSPECCION__CASOS_NO_RESUELTOS (estado actual)
#   Problema = NO_MATCH (1 token) o NO_MATCH_2TOKEN (2 tokens)
# ============================================================

mask_problem = merged["id_conllu_status"].isin(["NO_MATCH", "NO_MATCH_2TOKEN"])

problematic = merged.loc[
    mask_problem,
    ["id_archivo", "id_ud", "linea_n", "fenomeno", "forma_original", "forma_resultante", "id_conllu_status"]
].copy()

print("Total problemáticos:", len(problematic))
print(problematic["id_conllu_status"].value_counts(dropna=False))
problematic.head(30)

Total problemáticos: 20
id_conllu_status
NO_MATCH_2TOKEN    12
NO_MATCH            8
Name: count, dtype: int64


,id_archivo,id_ud,linea_n,fenomeno,forma_original,forma_resultante,id_conllu_status
78,001_003_001_002_002_012_002,UD00227,227,VARIANTE_LEXICA_CORCHETES,entonce,entonces,NO_MATCH
262,008_001_001_001_001_033_001,UD00015,15,VARIANTE_LEXICA_CORCHETES,preciosia,preciosa?,NO_MATCH
267,008_001_001_001_001_033_001,UD00053,53,VARIANTE_LEXICA_CORCHETES,el,te?,NO_MATCH
273,008_001_001_001_001_033_001,UD00154,154,VARIANTE_LEXICA_CORCHETES,po,pues?,NO_MATCH
1109,011_005_002_001_001_002_012,UD00569,569,VARIANTE_LEXICA_CORCHETES,ia,ya,NO_MATCH
1133,011_005_002_001_001_002_012,UD00600,600,VARIANTE_LEXICA_CORCHETES,io,yo,NO_MATCH
1610,014_004_001_002_001_013_001,UD00252,252,APOSTROFO,p’al,para el,NO_MATCH_2TOKEN
1634,014_004_001_002_001_013_001,UD00261,261,NORMALIZACION_LEXICA,arroxar,enrojar,NO_MATCH
1639,014_004_001_002_001_013_001,UD00262,262,NORMALIZACION_LEXICA,arroxar,enrojar,NO_MATCH
1724,014_004_001_002_001_013_001,UD00304,304,NORMALIZACION_LEXICA,conta-ys,conta ys,NO_MATCH_2TOKEN


In [ ]:
for _, r in problematic.head(10).iterrows():
    toks = sent_map.get((r["id_archivo"], r["id_ud"]))
    print(r["forma_resultante"], "->",
          [w for _, w in toks] if toks else "NO SENT")
    print("-----")

entonces -> ['pero', 'la', 'raíz', 'siempre', 'era', 'la', 'misma', 'entoncesno', 'se', 'era', 'tan', 'complicado', 'para', 'para', 'entender']
-----
preciosa? -> ['cualquier', 'tipo', 'de', 'palabra', 'preciosa', '?', 'aprenden', 'los', 'chicos', 'en', 'la', 'escuela']
-----
te? -> ['pero', 'no', 'no', 'hay', 'comunicación', 'así', 'el', 'el', 'guaraní', 'el', 'de', 'guaraní', 'te', '?', 'como', 'dicen', 'algunos', 'no', 'hay', 'ahí']
-----
pues? -> ['vos', 'sabés', 'pues', '?', 'que', 'si', 'va', 'la', 'be', 'o', 'la', 'o', 'la', 'hache', 'o', 'la', 'i']
-----
encargarles -> ['son', 'gente', 'menores', 'son', 'gente', 'grandes', 'debe', 'usted', 'di', 'encargar', 'les', 'que', 'no']
-----
platicarle -> ['de', 'de', 'valor', 'cuando', 'ven', 'la', 'gente', 'dale', 'pues', 'platicar', 'le', '¿', 'verdad', '?']
-----
ya -> ['ni', 'a', 'el', 'sol', 'el', 'sol', 'hay', 'que', 'agradecer', 'que', 'ya', 'salió', 'otro', 'nuevo', 'día', 'y', 'ya', 'empezó', 'y', 'a']
-----
yo -> ['¿', 'por',

#**PARTE III**

##**Conversión a CoNLL-U Plus**

###**III.a) Encabezado y extracción NER**
A continuación, se ha definido el encabezado del archivo CoNLL-U Plus mediante la especificación de las columnas adicionales del COREC y se ha implementado una función auxiliar que extrae la anotación NER desde el campo MISC (si está presente), la devuelve como columna separada (COREC:NER) y deja MISC sin el atributo NER para evitar duplicidades.

In [ ]:
# ============================================================
# CONVERSION__CONLLU_A_CONLLUP
# ============================================================

PLUS_HEADER = "# global.columns = ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC COREC:NER COREC:DISFL COREC:FORM_ORIG\n"


In [ ]:
#============================================================
# _MISC__EXTRACCION_NER_Y_LIMPIEZA
# ============================================================

def _extract_ner_and_clean_misc(misc: str) -> tuple[str, str]:
    """Devuelve (ner, misc_sin_ner)."""
    ner = "_"
    parts = [] if misc == "_" else misc.split("|")
    kept: list[str] = []
    for p in parts:
        key, *rest = p.split("=", 1)
        if rest and key.lower() == "ner":
            ner = rest[0] or "_"
        else:
            kept.append(p)
    misc_clean = "|".join(kept) if kept else "_"
    return ner, misc_clean


###**III.b) Generación mediante consulta O(1) de FORM_ORIG**

Con la siguiente función se ha añadido la columna COREC:FORM_ORIG. Para ello, se ha recorrido el archivo línea a línea, se han conservado los comentarios y los separadores de oración, se ha reconstruido el identificador oracional `id_ud` a partir de `# sent_id`, y para cada token real (ID entero) se ha consultado `form_orig_idx` mediante la clave `(id_archivo, id_ud, id_conllu)`. De este modo, se ha recuperado la forma_original correspondiente.

In [ ]:
def conllu_to_conllup(conllu_text: str, id_archivo: str, form_orig_idx) -> str:
    """
    Convierte CoNLL-U a CoNLL-U Plus añadiendo COREC:FORM_ORIG.

    form_orig_idx debe permitir:
        form_orig_idx.get((id_archivo, id_ud, id_conllu_int)) -> forma_original

    Si no hay forma_original para un token, se escribe "_" (o puedes usar FORM).
    """
    out: list[str] = [PLUS_HEADER]
    current_ud: str | None = None

    for line in conllu_text.splitlines():
        if not line.strip():
            out.append("")
            continue

        if line.startswith("#"):
            out.append(line)
            if line.startswith("# sent_id"):
                sent = line.split("=", 1)[1].strip()
                current_ud = f"UD{(int(sent) + 1):05d}" if sent.isdigit() else None
            continue

        cols = line.split("\t")
        if len(cols) != 10 or current_ud is None:
            out.append(line)
            continue

        ID = cols[0]

        # MWT o empty nodes: se dejan tal cual (sin COREC:FORM_ORIG)
        if "-" in ID or "." in ID or (not ID.isdigit()):
            out.append(line)
            continue

        key = (id_archivo, current_ud, int(ID))
        form_orig = form_orig_idx.get(key, "_")
        out.append(_expand_to_conllup(cols, form_orig))

    return "\n".join(out)

###**III.c) Expansión de filas CoNLL-U a CoNLL-U Plus (NER, DISFL y FORM_ORIG)**

A través de esta función se ha transformado cada fila CoNLL-U (10 columnas) en una fila CoNLL-U Plus (13 columnas).
Para ello se ha extraido la etiqueta NER desde MISC y se ha limpiado dicho campo para evitar duplicidades. Además, se ha inicializado COREC:DISFL con _ y se ha añadido COREC:FORM_ORIG con la forma original recibida.

In [ ]:
# ============================================================
# LINEA__CONSTRUCCION_CONLLUP
# ============================================================

def _expand_to_conllup(cols10: list[str], form_orig: str) -> str:
    """
    Expande una línea CoNLL-U (10 columnas) a CoNLL-U Plus (13 columnas),
    añadiendo COREC:NER, COREC:DISFL y COREC:FORM_ORIG.
    """
    ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC = cols10
    ner, misc_clean = _extract_ner_and_clean_misc(MISC)
    disfl = "_"
    form_orig_final = form_orig if form_orig not in ("", "_", None) else FORM
    return "\t".join([
        ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS,
        misc_clean, ner, disfl, form_orig_final
    ])


###**III.d) Construcción de FORM_ORIG_IDX**

Se ha creado posteriormente un diccionario de consulta O(1) que ha asignado a cada token su `forma_original` mediante la clave `(id_archivo, id_ud, id_conllu)`.

Para elo se han recorrido las filas del DataFrame `merged` con estado de alineamiento `OK`.

Cuando `id_conllu` se corresponde con un único token, la `forma_original` se ha asignado directamente. En los casos en que `id_conllu` representa un *span* `a-b`, el rango se ha expandido explícitamente y la misma `forma_original` se ha asignado a todos los tokens comprendidos entre `a` y `b`. De este modo, es posible proyectar `COREC:FORM_ORIG` token a token en la conversión al formato CoNLL-U Plus.

In [ ]:
# ============================================================
# BUILD_FORM_ORIG_IDX__O1 (incluye spans a-b)
# FORM_ORIG_IDX[(id_archivo,id_ud,id_conllu_int)] = forma_original
# ============================================================
FORM_ORIG_IDX = {}

ok_mask = merged["id_conllu_status"].astype(str).str.startswith("OK")

for _, r in merged.loc[ok_mask, ["id_archivo","id_ud","id_conllu","forma_original"]].iterrows():
    a = str(r["id_archivo"]).strip()
    u = str(r["id_ud"]).strip()
    idc = str(r["id_conllu"]).strip()
    fo = str(r["forma_original"]).strip()

    # token simple
    if idc.isdigit():
        key = (a, u, int(idc))
        if key not in FORM_ORIG_IDX:
            FORM_ORIG_IDX[key] = fo
        continue

    # span a-b
    if "-" in idc:
        left, right = idc.split("-", 1)
        if left.isdigit() and right.isdigit():
            a_id, b_id = int(left), int(right)
            for t in range(a_id, b_id + 1):
                key = (a, u, t)
                if key not in FORM_ORIG_IDX:
                    FORM_ORIG_IDX[key] = fo

print("OK FORM_ORIG_IDX keys:", len(FORM_ORIG_IDX))

OK FORM_ORIG_IDX keys: 2477


###**III.e) Conversión por archivo: CoNLL-U a CoNLL-U Plus**

Finalmente, se ha convertido cada archivo .conllu generado previamente a formato .conllup.
Para cada fichero se ha recuperado el identificador de archivo  y se ha aplicado `conllu_to_conllup`.
El resultado se ha guardado en el directorio de salida `conllup_out`.

In [ ]:
# ============================================================
# APLICAR__CONVERSION__CONLLU_A_CONLLUP__POR_ARCHIVO
# ============================================================

CONLLUP_DIR = Path("/content/conllup_out")
CONLLUP_DIR.mkdir(parents=True, exist_ok=True)

def out_conllup_path(input_conllu_path: Path) -> Path:
    return CONLLUP_DIR / f"{input_conllu_path.stem}.conllup"

conllu_files = sorted(Path("/content/conllu_out").glob("*.conllu"))

for i, conllu_path in enumerate(conllu_files, start=1):

    conllu_text = conllu_path.read_text(encoding="utf-8")


    id_archivo = id_archivo_base_from_conllu_path(conllu_path)

    conllup_text = conllu_to_conllup(
        conllu_text,
        id_archivo=id_archivo,
        form_orig_idx=FORM_ORIG_IDX
    )

    out_path = out_conllup_path(conllu_path)
    out_path.write_text(conllup_text, encoding="utf-8")

    print(f"[{i}/{len(conllu_files)}] Convertido:", conllu_path.name)

[1/11] Convertido: 001_003_001_002_002_012_002_seg_normas_1_normas_2 (15).conllu
[2/11] Convertido: 002_007_001_002_002_002_006_seg_normas_1_normas_2 (6).conllu
[3/11] Convertido: 003_006_004_001_001_029_006_seg_normas_1_normas_2 (3).conllu
[4/11] Convertido: 004_006_001_001_002_032_001_seg_normas_1_normas_2 (15).conllu
[5/11] Convertido: 005_004_002_003_002_001_001_seg_normas_1_normas_2 (3).conllu
[6/11] Convertido: 008_001_001_001_001_033_001_seg_normas_1_normas_2 (9).conllu
[7/11] Convertido: 010_006_002_001_002_014_008_seg_normas_1_normas_2 (5).conllu
[8/11] Convertido: 011_005_002_001_001_002_012_seg_normas_1_normas_2 (1).conllu
[9/11] Convertido: 012_001_002_001_001_034_004_seg_normas_1_normas_2 (1).conllu
[10/11] Convertido: 013_002_001_001_002_035_001_seg_normas_1_normas_2.conllu
[11/11] Convertido: 014_004_001_002_001_013_001_seg_normas_1_normas_2 (15).conllu
